# Sahil Bajaj — Qwen3-Coder 30B-A3B Q4_K_M

**Author:** Sahil Bajaj

# Qwen3-Coder 30B-A3B Q4_K_M — Kaggle 2×T4 + Ollama + Claude Code

Target model:

`qwen3-coder:30b-a3b-q4_K_M`

Designed for a Kaggle session with **2× NVIDIA Tesla T4**.

This notebook intentionally does **not** use Qwen3.8, MTP, or the old Qwen3.8 GGUF.

**Notebook author:** Sahil Bajaj


## 0. Kaggle configuration

**Notebook author:** Sahil Bajaj


In [ ]:
# Kaggle:
# Settings → Accelerator → GPU T4 ×2
# Settings → Internet → ON

# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

import os
import time
import json
import re
import shutil
import subprocess
from pathlib import Path

MODEL = "qwen3-coder:30b-a3b-q4_K_M"
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODELS = "/kaggle/working/ollama-models"

# Practical context for 2×T4.
NUM_CTX = 65536

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

Path(OLLAMA_MODELS).mkdir(parents=True, exist_ok=True)

print("MODEL:", MODEL)
print("OLLAMA:", OLLAMA_URL)
print("CONTEXT:", NUM_CTX)
print("MODEL STORE:", OLLAMA_MODELS)

## 1. Verify the two T4 GPUs

**Notebook author:** Sahil Bajaj


In [ ]:
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

subprocess.run([
    "bash", "-lc",
    "nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv"
], check=True)

## 2. Install required system packages

**Notebook author:** Sahil Bajaj


In [ ]:
# zstd is required by the current Ollama installer on this environment.
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

subprocess.run(
    ["bash", "-lc",
     "apt-get update -qq && apt-get install -y -qq zstd curl ca-certificates"],
    check=True
)

print("✓ zstd installed")
print("✓ curl installed")

## 3. Install Ollama

**Notebook author:** Sahil Bajaj


In [ ]:
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

OLLAMA_BIN = shutil.which("ollama")

if not OLLAMA_BIN:
    print("Installing Ollama...")
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True
    )
    OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"

print("Ollama binary:", OLLAMA_BIN)
print(subprocess.check_output([OLLAMA_BIN, "--version"], text=True).strip())

## 4. START SERVER — Ollama + download Qwen3-Coder

**Notebook author:** Sahil Bajaj


In [ ]:
# Start Ollama if it is not running.
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

def ollama_running():
    try:
        import urllib.request
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=3).close()
        return True
    except Exception:
        return False

if not ollama_running():
    ollama_log = open("/kaggle/working/ollama.log", "a")
    ollama_process = subprocess.Popen(
        [OLLAMA_BIN, "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy()
    )

    for _ in range(30):
        if ollama_running():
            break
        time.sleep(1)

    if not ollama_running():
        print(Path("/kaggle/working/ollama.log").read_text(errors="ignore")[-5000:])
        raise RuntimeError("Ollama did not start.")

print("✓ Ollama is running")

# Pull the exact Qwen3-Coder model.
models = subprocess.run(
    [OLLAMA_BIN, "list"],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
    check=True
).stdout

if MODEL not in models:
    print(f"Downloading {MODEL} ...")
    subprocess.run(
        [OLLAMA_BIN, "pull", MODEL],
        env=os.environ.copy(),
        check=True
    )
else:
    print("✓ Model already downloaded")

print("\nInstalled models:")
subprocess.run([OLLAMA_BIN, "list"], env=os.environ.copy(), check=True)

## 5. Load Qwen3-Coder and verify GPU usage

**Notebook author:** Sahil Bajaj


In [ ]:
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

import requests

payload = {
    "model": MODEL,
    "messages": [{
        "role": "user",
        "content": "Reply with exactly: Qwen3-Coder is working."
    }],
    "stream": False,
    "options": {
        "num_ctx": NUM_CTX
    }
}

r = requests.post(
    f"{OLLAMA_URL}/v1/chat/completions",
    json=payload,
    timeout=600
)
r.raise_for_status()

print(r.json()["choices"][0]["message"]["content"])

print("\n--- Ollama loaded model ---")
subprocess.run([OLLAMA_BIN, "ps"], env=os.environ.copy(), check=True)

print("\n--- GPU status ---")
subprocess.run(["bash", "-lc", "nvidia-smi"], check=True)

## 6. Test Anthropic Messages API locally

**Notebook author:** Sahil Bajaj


In [ ]:
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

payload = {
    "model": MODEL,
    "max_tokens": 128,
    "messages": [{
        "role": "user",
        "content": "Reply with exactly: Anthropic API is working."
    }]
}

headers = {
    "content-type": "application/json",
    "x-api-key": "ollama",
    "anthropic-version": "2023-06-01"
}

r = requests.post(
    f"{OLLAMA_URL}/v1/messages",
    headers=headers,
    json=payload,
    timeout=600
)
r.raise_for_status()

data = r.json()
print(json.dumps(data, indent=2)[:5000])
print("\n✓ Anthropic Messages API is working locally.")

## 7. Benchmark

**Notebook author:** Sahil Bajaj


In [ ]:
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

import time

prompt = """You are a coding assistant.
Explain how to debounce a React search input before making an API request.
Give a concise TypeScript example."""

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": prompt}],
    "stream": False,
    "options": {
        "num_ctx": NUM_CTX,
        "temperature": 0
    }
}

start = time.perf_counter()
r = requests.post(
    f"{OLLAMA_URL}/v1/chat/completions",
    json=payload,
    timeout=600
)
wall = time.perf_counter() - start
r.raise_for_status()

data = r.json()
print(data["choices"][0]["message"]["content"])

usage = data.get("usage", {})
generated = usage.get("completion_tokens")

print("\n--- Benchmark ---")
print("Generated tokens:", generated)
print("Wall time:", round(wall, 2), "s")

if generated:
    print("Approx speed:", round(generated / wall, 2), "tokens/s")

## 8. Install Cloudflared

**Notebook author:** Sahil Bajaj


In [ ]:
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

CLOUDFLARED_BIN = shutil.which("cloudflared")

if not CLOUDFLARED_BIN:
    subprocess.run(
        ["bash", "-lc", """
        set -e
        ARCH=$(dpkg --print-architecture)
        case "$ARCH" in
            amd64) CF_ARCH="amd64" ;;
            arm64) CF_ARCH="arm64" ;;
            *) echo "Unsupported architecture: $ARCH"; exit 1 ;;
        esac

        curl -L --fail --silent --show-error \
          -o /usr/local/bin/cloudflared \
          "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-${CF_ARCH}"

        chmod +x /usr/local/bin/cloudflared
        """],
        check=True
    )
    CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"

print(subprocess.check_output(
    [CLOUDFLARED_BIN, "--version"],
    text=True
).strip())

## 9. Public tunnel — optional first-time test

**Notebook author:** Sahil Bajaj


In [ ]:
# This cell is optional. The bottom ON block can create the tunnel later.
# If you run it now, the ON block will replace it with a fresh tunnel.

# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

print("Cloudflared installed:", CLOUDFLARED_BIN)
print("The permanent daily ON/OFF controls are at the bottom.")

# SERVER CONTROLS

After the first-time setup, you can use only the two blocks at the bottom.

- **ON:** self-checks/install missing Ollama + cloudflared, starts Ollama, pulls the exact Qwen3-Coder model if missing, starts Cloudflare, tests the public Anthropic API, and prints `settings.json`.
- **OFF:** stops Ollama + Cloudflare without shutting down the Kaggle notebook.

If Kaggle gives you a completely fresh runtime, **ON is designed to recover automatically**. The model may need to be downloaded again if its files are not present in persistent storage.

**Notebook author:** Sahil Bajaj


In [ ]:
# ==========================================================
# QWEN3-CODER — SERVER ON (AUTO-RECOVERY)
# ==========================================================

# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

import os, re, json, time, shutil, subprocess, socket
from pathlib import Path
import requests

MODEL = "qwen3-coder:30b-a3b-q4_K_M"
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODELS = "/kaggle/working/ollama-models"

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

# ----------------------------------------------------------
# 0. Recover required command-line tools if this is a
#    completely fresh Kaggle runtime.
# ----------------------------------------------------------

def command_exists(name):
    return shutil.which(name) is not None

if not command_exists("curl") or not command_exists("zstd"):
    print("[0/5] Installing missing system dependencies...")
    subprocess.run(
        ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq curl ca-certificates zstd"],
        check=True
    )

OLLAMA_BIN = shutil.which("ollama")

if not OLLAMA_BIN:
    print("[0/5] Ollama is missing. Installing Ollama...")
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True
    )
    OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"

if not Path(OLLAMA_BIN).exists():
    raise RuntimeError(f"Ollama installation finished, but binary was not found: {OLLAMA_BIN}")

CLOUDFLARED_BIN = shutil.which("cloudflared")

if not CLOUDFLARED_BIN:
    print("[0/5] cloudflared is missing. Installing cloudflared...")
    subprocess.run(
        ["bash", "-lc", """
        set -e
        ARCH=$(dpkg --print-architecture)
        case "$ARCH" in
            amd64) CF_ARCH="amd64" ;;
            arm64) CF_ARCH="arm64" ;;
            *) echo "Unsupported architecture: $ARCH"; exit 1 ;;
        esac

        curl -L --fail --silent --show-error \
          -o /usr/local/bin/cloudflared \
          "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-${CF_ARCH}"

        chmod +x /usr/local/bin/cloudflared
        """],
        check=True
    )
    CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"

if not Path(CLOUDFLARED_BIN).exists():
    raise RuntimeError(f"cloudflared installation finished, but binary was not found: {CLOUDFLARED_BIN}")

print("    Ollama:", OLLAMA_BIN)
print("    cloudflared:", CLOUDFLARED_BIN)

# ----------------------------------------------------------
# 1. Start Ollama
# ----------------------------------------------------------

def ollama_running():
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except Exception:
        return False

if not ollama_running():
    print("[1/5] Starting Ollama...")
    ollama_log_path = "/kaggle/working/ollama-on.log"
    ollama_log = open(ollama_log_path, "a")

    ollama_process = subprocess.Popen(
        [OLLAMA_BIN, "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy()
    )

    for _ in range(45):
        if ollama_running():
            break
        time.sleep(1)

    if not ollama_running():
        log = Path(ollama_log_path).read_text(errors="ignore")
        raise RuntimeError("Ollama failed to start.\n\n" + log[-8000:])
else:
    print("[1/5] Ollama already running.")

# ----------------------------------------------------------
# 2. Ensure exact Qwen3-Coder model exists
# ----------------------------------------------------------

models = subprocess.run(
    [OLLAMA_BIN, "list"],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
    check=True
).stdout

if MODEL not in models:
    print(f"[2/5] {MODEL} is missing. Downloading it now...")
    subprocess.run(
        [OLLAMA_BIN, "pull", MODEL],
        env=os.environ.copy(),
        check=True
    )
else:
    print("[2/5] Qwen3-Coder already downloaded.")

print("    Model:", MODEL)

# ----------------------------------------------------------
# 3. Start a fresh Cloudflare Quick Tunnel
# ----------------------------------------------------------

# If ON was already run earlier in this same notebook, stop only
# the tunnel process that this notebook created.
if "cloudflared_process" in globals():
    try:
        if cloudflared_process.poll() is None:
            cloudflared_process.terminate()
            cloudflared_process.wait(timeout=10)
    except Exception:
        try:
            cloudflared_process.kill()
        except Exception:
            pass

tunnel_log = "/kaggle/working/qwen3coder-cloudflared.log"
Path(tunnel_log).write_text("")

print("[3/5] Starting Cloudflare Quick Tunnel...")

cloudflared_process = subprocess.Popen(
    [
        CLOUDFLARED_BIN,
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
        "--url", OLLAMA_URL,
        "--http-host-header", "localhost:11434"
    ],
    stdout=open(tunnel_log, "a"),
    stderr=subprocess.STDOUT,
    text=True
)

PUBLIC_OLLAMA_URL = None

for _ in range(90):
    time.sleep(1)

    if cloudflared_process.poll() is not None:
        log = Path(tunnel_log).read_text(errors="ignore")
        raise RuntimeError(
            "cloudflared exited before creating the tunnel.\n\n" + log[-10000:]
        )

    log = Path(tunnel_log).read_text(errors="ignore")
    urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)

    if urls:
        candidate = urls[-1].rstrip("/")
        hostname = candidate.split("://", 1)[1]

        # Quick Tunnel DNS can take a little time to propagate.
        for _dns_try in range(45):
            try:
                socket.gethostbyname(hostname)
                PUBLIC_OLLAMA_URL = candidate
                break
            except socket.gaierror:
                time.sleep(2)

        if PUBLIC_OLLAMA_URL:
            break

if not PUBLIC_OLLAMA_URL:
    print(Path(tunnel_log).read_text(errors="ignore")[-12000:])
    raise RuntimeError("Cloudflare tunnel URL was not usable.")

print("    Public URL:", PUBLIC_OLLAMA_URL)

# ----------------------------------------------------------
# 4. Test the public Anthropic-compatible API
# ----------------------------------------------------------

print("[4/5] Testing public Anthropic Messages API...")

payload = {
    "model": MODEL,
    "max_tokens": 128,
    "messages": [{
        "role": "user",
        "content": "Reply with exactly: Qwen3-Coder public API is working."
    }]
}

headers = {
    "content-type": "application/json",
    "x-api-key": "ollama",
    "anthropic-version": "2023-06-01"
}

last_error = None

for attempt in range(1, 11):
    try:
        r = requests.post(
            f"{PUBLIC_OLLAMA_URL}/v1/messages",
            headers=headers,
            json=payload,
            timeout=120
        )
        r.raise_for_status()
        api_data = r.json()
        print(f"    ✓ Public Anthropic API works (attempt {attempt})")
        break
    except Exception as e:
        last_error = e
        print(f"    Attempt {attempt}/10 failed; retrying in 3 seconds...")
        time.sleep(3)
else:
    raise RuntimeError(
        f"Public tunnel was created but the API could not be reached: {last_error}"
    )

# ----------------------------------------------------------
# 5. Generate Claude Code settings.json
# ----------------------------------------------------------

settings = {
    "env": {
        "ANTHROPIC_BASE_URL": PUBLIC_OLLAMA_URL,
        "ANTHROPIC_AUTH_TOKEN": "ollama",
        "ANTHROPIC_MODEL": MODEL,
        "CLAUDE_CODE_DISABLE_UNKNOWN_MODEL_WINDOW_ENFORCEMENT": "1"
    },
    "model": MODEL,
    "theme": "dark"
}

print("\n[5/5] COPY THIS INTO CLAUDE CODE settings.json")
print("=" * 60)
print(json.dumps(settings, indent=2))
print("=" * 60)

print("\nSERVER STATUS: ON — Sahil Bajaj")
print("Model:", MODEL)
print("API:", PUBLIC_OLLAMA_URL + "/v1")
print("Context configured in Ollama notebook:", "65536 tokens")


## SERVER OFF

Use this only when you want to stop Ollama and the Cloudflare tunnel while keeping the Kaggle notebook/runtime alive.

**Notebook author:** Sahil Bajaj


In [ ]:
# ==========================================================
# QWEN3-CODER — SERVER OFF
# ==========================================================

# Stop Cloudflare (only the tunnel started by this notebook)
# Notebook author: Sahil Bajaj
NOTEBOOK_AUTHOR = "Sahil Bajaj"

if "cloudflared_process" in globals():
    try:
        if cloudflared_process.poll() is None:
            cloudflared_process.terminate()
            cloudflared_process.wait(timeout=10)
    except Exception:
        try:
            cloudflared_process.kill()
        except Exception:
            pass

# Stop Ollama
if "ollama_process" in globals():
    try:
        if ollama_process.poll() is None:
            ollama_process.terminate()
            ollama_process.wait(timeout=10)
    except Exception:
        try:
            ollama_process.kill()
        except Exception:
            pass

# Clean up processes started by earlier notebook executions.
subprocess.run(
    ["bash", "-lc", "pkill -f 'cloudflared tunnel' || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
subprocess.run(
    ["bash", "-lc", "pkill -f 'ollama serve' || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("✓ Ollama OFF")
print("✓ Cloudflare OFF")
print("✓ Kaggle session remains ON")